# Phase 1: Data Preparation & Translation\n\nLoad InstructPix2Pix dataset, sample a subset, save images, and translate edit prompts to Hindi, Bangla, and Nepali using MarianMT.\n\n**Output:** `phase1_dataset.parquet` + `images/{original,edited}/`

In [ ]:
# Cell 1: Install dependencies
!pip install -q datasets sacrebleu sentencepiece transformers torch Pillow pandas

In [ ]:
# Cell 2: Config & Utils
import os
import sys
import json
import re
import numpy as np
import pandas as pd
import torch
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm

# ── Path Detection ──
IS_KAGGLE = os.path.exists("/kaggle/working")
OUTPUT_DIR = "/kaggle/working" if IS_KAGGLE else "./output"
INPUT_DIR = "/kaggle/input" if IS_KAGGLE else "./output"

# ── Dataset ──
HF_DATASET_NAME = "timbrooks/instructpix2pix-clip-filtered"
SUBSET_SIZE = 5000
RANDOM_SEED = 42

# ── Translation (MarianMT) ──
TRANSLATION_MODELS = {
    "hi": "Helsinki-NLP/opus-mt-en-hi",
    "bn": "Helsinki-NLP/opus-mt-en-mul",
    "ne": "Helsinki-NLP/opus-mt-en-mul",
}
BACK_TRANSLATION_MODELS = {
    "hi": "Helsinki-NLP/opus-mt-hi-en",
    "bn": "Helsinki-NLP/opus-mt-mul-en",
    "ne": "Helsinki-NLP/opus-mt-mul-en",
}
BLEU_THRESHOLD = 0.65

LANGUAGE_TAGS = {
    "bn": ">>ben<<",
    "ne": ">>npi<<",
}

# Paths
IMAGE_DIR_ORIG = os.path.join(OUTPUT_DIR, "images", "original")
IMAGE_DIR_EDIT = os.path.join(OUTPUT_DIR, "images", "edited")
os.makedirs(IMAGE_DIR_ORIG, exist_ok=True)
os.makedirs(IMAGE_DIR_EDIT, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"IS_KAGGLE: {IS_KAGGLE}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"Image dirs: {IMAGE_DIR_ORIG}, {IMAGE_DIR_EDIT}")

In [ ]:
# Cell 3: Load dataset from HuggingFace
from datasets import load_dataset

print(f"Loading dataset: {HF_DATASET_NAME}")
dataset = load_dataset(HF_DATASET_NAME, split="train")
print(f"Full dataset size: {len(dataset)}")
print(f"Columns: {dataset.column_names}")
print(f"Sample: {dataset[0]['edit_prompt']}")

In [ ]:
# Cell 4: Stratified sampling by action verb
import random

def extract_action_verb(prompt):
    """Extract first word as action verb for stratification."""
    words = prompt.strip().lower().split()
    if words:
        return words[0]
    return "unknown"

# Group by action verb
prompts = [dataset[i]["edit_prompt"] for i in range(len(dataset))]
verb_to_indices = {}
for i, p in enumerate(prompts):
    verb = extract_action_verb(p)
    verb_to_indices.setdefault(verb, []).append(i)

print(f"Found {len(verb_to_indices)} unique action verbs")
print(f"Top 10 verbs: {sorted(verb_to_indices.keys(), key=lambda v: -len(verb_to_indices[v]))[:10]}")

# Proportional stratified sampling
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

total = len(dataset)
selected_indices = []

for verb, indices in verb_to_indices.items():
    proportion = len(indices) / total
    n_samples = max(1, int(proportion * SUBSET_SIZE))
    sampled = random.sample(indices, min(n_samples, len(indices)))
    selected_indices.extend(sampled)

# Trim or pad to exact SUBSET_SIZE
random.shuffle(selected_indices)
if len(selected_indices) > SUBSET_SIZE:
    selected_indices = selected_indices[:SUBSET_SIZE]
elif len(selected_indices) < SUBSET_SIZE:
    remaining = list(set(range(total)) - set(selected_indices))
    extra = random.sample(remaining, SUBSET_SIZE - len(selected_indices))
    selected_indices.extend(extra)

selected_indices.sort()
print(f"Selected {len(selected_indices)} samples")

In [ ]:
# Cell 5: Save images to disk
records = []

for new_idx, ds_idx in enumerate(tqdm(selected_indices, desc="Saving images")):
    sample = dataset[ds_idx]
    sample_id = f"{new_idx:06d}"

    orig_path = os.path.join(IMAGE_DIR_ORIG, f"{sample_id}.jpg")
    edit_path = os.path.join(IMAGE_DIR_EDIT, f"{sample_id}.jpg")

    # Save original image
    orig_img = sample["original_image"]
    if isinstance(orig_img, Image.Image):
        orig_img.convert("RGB").save(orig_path, quality=95)
    
    # Save edited image
    edit_img = sample["edited_image"]
    if isinstance(edit_img, Image.Image):
        edit_img.convert("RGB").save(edit_path, quality=95)

    records.append({
        "sample_id": sample_id,
        "dataset_index": ds_idx,
        "original_image_path": f"images/original/{sample_id}.jpg",
        "edited_image_path": f"images/edited/{sample_id}.jpg",
        "edit_prompt": sample["edit_prompt"],
    })

df = pd.DataFrame(records)
print(f"Created DataFrame with {len(df)} rows")
print(df.head())

In [ ]:
# Cell 6: MarianMT Translator class
from transformers import MarianMTModel, MarianTokenizer

class MarianTranslator:
    """Handles forward and back translation with MarianMT models."""
    
    def __init__(self, target_lang, device="cpu"):
        self.target_lang = target_lang
        self.device = device
        self.lang_tag = LANGUAGE_TAGS.get(target_lang, None)
        
        # Load forward model (en -> target)
        fwd_model_name = TRANSLATION_MODELS[target_lang]
        print(f"Loading forward model: {fwd_model_name} for {target_lang}")
        self.fwd_tokenizer = MarianTokenizer.from_pretrained(fwd_model_name)
        self.fwd_model = MarianMTModel.from_pretrained(fwd_model_name).to(device)
        self.fwd_model.eval()
        
        # Load backward model (target -> en)
        bwd_model_name = BACK_TRANSLATION_MODELS[target_lang]
        print(f"Loading backward model: {bwd_model_name} for {target_lang}")
        self.bwd_tokenizer = MarianTokenizer.from_pretrained(bwd_model_name)
        self.bwd_model = MarianMTModel.from_pretrained(bwd_model_name).to(device)
        self.bwd_model.eval()
    
    def _add_lang_tag(self, texts):
        """Add language tag prefix for multilingual models."""
        if self.lang_tag:
            return [f"{self.lang_tag} {t}" for t in texts]
        return texts
    
    @torch.no_grad()
    def translate_batch(self, texts, batch_size=64):
        """Translate English texts to target language."""
        tagged_texts = self._add_lang_tag(texts)
        all_translations = []
        
        for i in range(0, len(tagged_texts), batch_size):
            batch = tagged_texts[i:i + batch_size]
            inputs = self.fwd_tokenizer(batch, return_tensors="pt",
                                         padding=True, truncation=True,
                                         max_length=128).to(self.device)
            outputs = self.fwd_model.generate(**inputs, max_length=128)
            decoded = self.fwd_tokenizer.batch_decode(outputs, skip_special_tokens=True)
            all_translations.extend(decoded)
        
        return all_translations
    
    @torch.no_grad()
    def back_translate_batch(self, texts, batch_size=64):
        """Translate target language texts back to English."""
        all_translations = []
        
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            inputs = self.bwd_tokenizer(batch, return_tensors="pt",
                                         padding=True, truncation=True,
                                         max_length=128).to(self.device)
            outputs = self.bwd_model.generate(**inputs, max_length=128)
            decoded = self.bwd_tokenizer.batch_decode(outputs, skip_special_tokens=True)
            all_translations.extend(decoded)
        
        return all_translations
    
    def cleanup(self):
        """Free GPU memory."""
        del self.fwd_model, self.bwd_model
        del self.fwd_tokenizer, self.bwd_tokenizer
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f"Cleaned up {self.target_lang} translator")

print("MarianTranslator class defined.")

In [ ]:
# Cell 7: Run translations for all languages
import sacrebleu

device = "cuda" if torch.cuda.is_available() else "cpu"
edit_prompts = df["edit_prompt"].tolist()

for lang in ["hi", "bn", "ne"]:
    print(f"\n{'='*60}")
    print(f"Translating to {lang}...")
    print(f"{'='*60}")
    
    translator = MarianTranslator(lang, device=device)
    
    # Forward translation
    translated = translator.translate_batch(edit_prompts)
    df[f"edit_prompt_{lang}"] = translated
    
    # Back translation for quality assessment
    back_translated = translator.back_translate_batch(translated)
    
    # Compute per-sentence BLEU scores
    bleu_scores = []
    for orig, bt in zip(edit_prompts, back_translated):
        try:
            score = sacrebleu.sentence_bleu(bt, [orig]).score / 100.0  # normalize to 0-1
        except Exception:
            score = 0.0
        bleu_scores.append(score)
    
    df[f"translation_confidence_{lang}"] = bleu_scores
    df[f"low_confidence_{lang}"] = [s < BLEU_THRESHOLD for s in bleu_scores]
    
    # Show examples
    print(f"\nSample translations ({lang}):")
    for i in range(min(3, len(df))):
        print(f"  EN: {edit_prompts[i]}")
        print(f"  {lang.upper()}: {translated[i]}")
        print(f"  Back: {back_translated[i]}")
        print(f"  BLEU: {bleu_scores[i]:.3f}")
        print()
    
    # Cleanup to free VRAM
    translator.cleanup()
    
    low_count = sum(df[f"low_confidence_{lang}"])
    print(f"{lang}: {low_count}/{len(df)} low confidence translations ({low_count/len(df)*100:.1f}%)")

print("\nTranslation complete!")

In [ ]:
# Cell 8: Save final parquet
output_path = os.path.join(OUTPUT_DIR, "phase1_dataset.parquet")
df.to_parquet(output_path, index=False)
print(f"Saved to {output_path}")
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nDataFrame info:")
print(df.dtypes)

In [ ]:
# Cell 9: Summary statistics
print("=" * 60)
print("PHASE 1 SUMMARY")
print("=" * 60)
print(f"Total samples: {len(df)}")
print(f"Images saved: {len(os.listdir(IMAGE_DIR_ORIG))} original, {len(os.listdir(IMAGE_DIR_EDIT))} edited")
print()

for lang in ["hi", "bn", "ne"]:
    low = df[f"low_confidence_{lang}"].sum()
    avg_conf = df[f"translation_confidence_{lang}"].mean()
    print(f"{lang.upper()}: avg confidence={avg_conf:.3f}, low confidence={low}/{len(df)} ({low/len(df)*100:.1f}%)")

print(f"\nAction verb distribution (top 10):")
verbs = df["edit_prompt"].apply(extract_action_verb)
print(verbs.value_counts().head(10))

print(f"\nSample translations:")
for i in range(min(5, len(df))):
    row = df.iloc[i]
    print(f"\n  EN: {row['edit_prompt']}")
    for lang in ["hi", "bn", "ne"]:
        print(f"  {lang.upper()}: {row[f'edit_prompt_{lang}']} (conf: {row[f'translation_confidence_{lang}']:.3f})")